<a href="https://colab.research.google.com/github/parthnayyar07/mausammix/blob/main/sih.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q xarray zarr gcsfs dask

import xarray as xr

OPTS = {"token": "anon"}
BASE = "gs://weatherbench2/datasets"

ds = xr.open_zarr(
    f"{BASE}/hres/2016-2022-0012-240x121_equiangular_with_poles_conservative.zarr",
    storage_options=OPTS,
)
print(ds)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.2 MB/s eta 0:00:00


/tmp/ipykernel_9398/2749843773.py:8: FutureWarning: In a future version, xarray will not decode the variable 'prediction_timedelta' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_zarr(


<xarray.Dataset> Size: 2TB
Dimensions:                   (time: 5114, prediction_timedelta: 41,
                               longitude: 240, latitude: 121, level: 13)
Coordinates:
  * time                      (time) datetime64[ns] 41kB 2016-01-01 ... 2022-...
  * prediction_timedelta      (prediction_timedelta) timedelta64[ns] 328B 00:...
  * longitude                 (longitude) float64 2kB 0.0 1.5 ... 357.0 358.5
  * latitude                  (latitude) float64 968B -90.0 -88.5 ... 88.5 90.0
  * level                     (level) int32 52B 50 100 150 200 ... 850 925 1000
Data variables: (12/16)
    10m_u_component_of_wind   (time, prediction_timedelta, longitude, latitude) float32 24GB dask.array<chunksize=(1, 8, 240, 121), meta=np.ndarray>
    10m_v_component_of_wind   (time, prediction_timedelta, longitude, latitude) float32 24GB dask.array<chunksize=(1, 8, 240, 121), meta=np.ndarray>
    10m_wind_speed            (time, prediction_timedelta, longitude, latitude) float32 24GB das

In [2]:
import xarray as xr
import numpy as np

OPTS = {"token": "anon"}
BASE = "gs://weatherbench2/datasets"
VAR  = "2m_temperature"

INDIA = dict(latitude=slice(5, 40), longitude=slice(65, 100))

def load_india(path, var=VAR):
    ds = xr.open_zarr(path, storage_options=OPTS, decode_timedelta=True)
    da = ds[var].sortby("latitude").sortby("longitude")
    return da.sel(**INDIA)

hres = load_india(f"{BASE}/hres/2016-2022-0012-240x121_equiangular_with_poles_conservative.zarr")
pangu = load_india(f"{BASE}/pangu/2018-2022_0012_240x121_equiangular_with_poles_conservative.zarr")
era5  = load_india(f"{BASE}/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")

for name, da in [("HRES", hres), ("Pangu", pangu), ("ERA5 truth", era5)]:
    print(f"\n=== {name} ===")
    print("dims:", dict(da.sizes))
    print("lat:", float(da.latitude.min()), "→", float(da.latitude.max()))
    print("lon:", float(da.longitude.min()), "→", float(da.longitude.max()))
    print("time:", str(da.time.min().values)[:10], "→", str(da.time.max().values)[:10])
    if "prediction_timedelta" in da.dims:
        lt = da.prediction_timedelta.values
        print("lead times:", lt[0], "...", lt[-1], f"({len(lt)} steps)")


=== HRES ===
dims: {'time': 5114, 'prediction_timedelta': 41, 'longitude': 23, 'latitude': 23}
lat: 5.999999999999996 → 38.99999999999999
lon: 65.99999999999999 → 99.0
time: 2016-01-01 → 2022-12-31
lead times: 0 nanoseconds ... 864000000000000 nanoseconds (41 steps)

=== Pangu ===
dims: {'time': 3652, 'prediction_timedelta': 40, 'longitude': 23, 'latitude': 23}
lat: 5.999999999999996 → 38.99999999999999
lon: 65.99999999999999 → 99.0
time: 2018-01-01 → 2022-12-31
lead times: 21600000000000 nanoseconds ... 864000000000000 nanoseconds (40 steps)

=== ERA5 truth ===
dims: {'time': 93544, 'longitude': 23, 'latitude': 23}
lat: 5.999999999999996 → 38.99999999999999
lon: 65.99999999999999 → 99.0
time: 1959-01-01 → 2023-01-10


In [3]:
!pip install -q zarr gcsfs

In [4]:
import xarray as xr, zarr, numpy as np, pandas as pd, matplotlib.pyplot as plt, os
print("xarray", xr.__version__, "| zarr", zarr.__version__)
print("engines:", list(xr.backends.list_engines()))   # 'zarr' must appear here

OPTS  = {"token": "anon"}
BASE  = "gs://weatherbench2/datasets"
VAR   = "2m_temperature"
INDIA = dict(latitude=slice(5, 40), longitude=slice(65, 100))

def load_india(path, var=VAR):
    ds = xr.open_zarr(path, storage_options=OPTS, decode_timedelta=True)
    return ds[var].sortby("latitude").sortby("longitude").sel(**INDIA)

hres  = load_india(f"{BASE}/hres/2016-2022-0012-240x121_equiangular_with_poles_conservative.zarr")
pangu = load_india(f"{BASE}/pangu/2018-2022_0012_240x121_equiangular_with_poles_conservative.zarr")
era5  = load_india(f"{BASE}/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")

print("loaded:", dict(hres.sizes), dict(pangu.sizes), dict(era5.sizes))

xarray 2025.12.0 | zarr 3.4.0
engines: ['h5netcdf', 'scipy', 'store', 'zarr']
loaded: {'time': 5114, 'prediction_timedelta': 41, 'longitude': 23, 'latitude': 23} {'time': 3652, 'prediction_timedelta': 40, 'longitude': 23, 'latitude': 23} {'time': 93544, 'longitude': 23, 'latitude': 23}


In [5]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/sih081'
os.makedirs(SAVE, exist_ok=True)
print(SAVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/sih081


In [ ]:
LEAD_DAYS = [1, 3, 5, 7, 10]
START, END = "2018-01-01", "2022-12-20"

def align(fc, truth, lead_days):
    lead = np.timedelta64(lead_days, "D")
    f = fc.sel(prediction_timedelta=lead).sel(time=slice(START, END))
    f = f.sel(time=f.time[f.time.dt.hour == 0])      # 00 UTC runs only
    t = truth.sel(time=f.time.values + lead)         # what actually happened
    t = t.assign_coords(time=f.time.values)          # label by init date
    return f, t

store = {}
for d in LEAD_DAYS:
    print(f"loading day {d} ...", end=" ")
    fh, th = align(hres,  era5, d)
    fp, _  = align(pangu, era5, d)
    fh, fp, th = xr.align(fh, fp, th, join="inner")
    store[d] = {"hres": fh.load(), "pangu": fp.load(), "truth": th.load()}
    print(f"{store[d]['hres'].sizes['time']} dates")

out = xr.Dataset({
    f"{k}_d{d}": store[d][k].drop_vars("prediction_timedelta", errors="ignore")
    for d in LEAD_DAYS for k in ("hres", "pangu", "truth")
})
out.to_netcdf(f"{SAVE}/india_t2m.nc")
print("saved to Drive")

loading day 1 ... 1815 dates
loading day 3 ... 1815 dates
loading day 5 ... 

In [ ]:
!pip install -q zarr gcsfs

In [ ]:
import xarray as xr, numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/sih081'

out = xr.open_dataset(f"{SAVE}/india_t2m.nc")
LEAD_DAYS = [1, 3, 5, 7, 10]
store = {d: {k: out[f"{k}_d{d}"] for k in ("hres", "pangu", "truth")} for d in LEAD_DAYS}
print(out)

In [ ]:
def rmse(f, t):
    w = np.cos(np.deg2rad(f.latitude))
    return float(np.sqrt(((f - t) ** 2).weighted(w).mean()))

TEST = slice("2020-01-01", "2022-12-20")

rows = []
for d in LEAD_DAYS:
    s  = store[d]
    tr = s["truth"].sel(time=TEST)
    rows.append({
        "lead_day": d,
        "HRES":  rmse(s["hres"].sel(time=TEST),  tr),
        "Pangu": rmse(s["pangu"].sel(time=TEST), tr),
    })

df = pd.DataFrame(rows)
df["best_model"] = np.where(df.HRES < df.Pangu, "HRES", "Pangu")
print(df.round(3))
df.to_csv(f"{SAVE}/results_baseline.csv", index=False)

plt.figure(figsize=(7, 4.5))
plt.plot(df.lead_day, df.HRES,  "o-", label="IFS HRES (physics)")
plt.plot(df.lead_day, df.Pangu, "s-", label="Pangu-Weather (AI)")
plt.xlabel("Lead time (days)"); plt.ylabel("RMSE (K)")
plt.title("2 m temperature forecast error over India, 2020–2022")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{SAVE}/chart1_baseline.png", dpi=200)
plt.show()

In [ ]:
BASE = "gs://weatherbench2/datasets"
OPTS = {"token": "anon"}
VAR, INDIA = "2m_temperature", dict(latitude=slice(5,40), longitude=slice(65,100))

def load_india(path, var=VAR):
    ds = xr.open_zarr(path, storage_options=OPTS, decode_timedelta=True)
    return ds[var].sortby("latitude").sortby("longitude").sel(**INDIA)

hres_t0 = load_india(f"{BASE}/hres_t0/2016-2022-6h-240x121_equiangular_with_poles_conservative.zarr")

def truth_for(fc_da, truth_da, lead_days):
    lead = np.timedelta64(lead_days, "D")
    t = truth_da.sel(time=fc_da.time.values + lead)
    return t.assign_coords(time=fc_da.time.values).load()

for d in LEAD_DAYS:
    store[d]["truth_hres"] = truth_for(store[d]["hres"], hres_t0, d)

rows = []
for d in LEAD_DAYS:
    s = store[d]
    rows.append({
        "lead_day": d,
        "HRES_vs_ERA5":     rmse(s["hres"].sel(time=TEST),  s["truth"].sel(time=TEST)),
        "HRES_vs_analysis": rmse(s["hres"].sel(time=TEST),  s["truth_hres"].sel(time=TEST)),
        "Pangu_vs_ERA5":    rmse(s["pangu"].sel(time=TEST), s["truth"].sel(time=TEST)),
    })
df_fair = pd.DataFrame(rows)
print(df_fair.round(3))
df_fair.to_csv(f"{SAVE}/results_fair.csv", index=False)

In [ ]:
def skill_weighted(f1, f2, truth, lead_days, window=30):
    e1 = ((f1 - truth) ** 2)
    e2 = ((f2 - truth) ** 2)
    shift = lead_days + 1                       # no future information
    m1 = e1.shift(time=shift).rolling(time=window, min_periods=10).mean()
    m2 = e2.shift(time=shift).rolling(time=window, min_periods=10).mean()
    w1 = (1/(m1+1e-6)) / ((1/(m1+1e-6)) + (1/(m2+1e-6)))
    w2 = 1 - w1
    return w1*f1 + w2*f2, w1

rows = []
for d in LEAD_DAYS:
    s  = store[d]
    bl, w1 = skill_weighted(s["hres"], s["pangu"], s["truth"], d)
    s["blend"], s["w_hres"] = bl, w1
    tr = s["truth"].sel(time=TEST)
    r_h = rmse(s["hres"].sel(time=TEST),  tr)
    r_p = rmse(s["pangu"].sel(time=TEST), tr)
    r_e = rmse(((s["hres"]+s["pangu"])/2).sel(time=TEST), tr)
    r_b = rmse(bl.sel(time=TEST), tr)
    rows.append({"lead_day": d, "HRES": r_h, "Pangu": r_p,
                 "EqualBlend": r_e, "SmartBlend": r_b,
                 "gain_vs_best_%": 100*(1 - r_b/min(r_h, r_p))})

df_b = pd.DataFrame(rows)
print(df_b.round(3))
df_b.to_csv(f"{SAVE}/results_blend.csv", index=False)

plt.figure(figsize=(7.5,4.8))
plt.plot(df_b.lead_day, df_b.HRES,       "o-",  label="IFS HRES (physics)")
plt.plot(df_b.lead_day, df_b.Pangu,      "s-",  label="Pangu-Weather (AI)")
plt.plot(df_b.lead_day, df_b.EqualBlend, "^--", label="Equal-weight blend")
plt.plot(df_b.lead_day, df_b.SmartBlend, "D-",  lw=2.5, color="crimson", label="MausamMix adaptive blend")
plt.xlabel("Lead time (days)"); plt.ylabel("RMSE (K)")
plt.title("Adaptive blending lowers 2 m temperature error over India (2020–2022)")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{SAVE}/chart2_blend.png", dpi=200)
plt.show()

In [ ]:
def bias_corrected_blend(f1, f2, truth, lead_days, window=30, power=1):
    shift = lead_days + 1
    def past(x): return x.shift(time=shift).rolling(time=window, min_periods=10).mean()
    b1, b2 = past(f1 - truth), past(f2 - truth)         # recent bias
    c1, c2 = f1 - b1, f2 - b2                            # de-biased forecasts
    m1, m2 = past((c1 - truth)**2), past((c2 - truth)**2)
    i1, i2 = 1/(m1+1e-6)**power, 1/(m2+1e-6)**power
    w1 = i1/(i1+i2)
    return w1*c1 + (1-w1)*c2, w1

rows = []
for d in LEAD_DAYS:
    s = store[d]
    tr = s["truth"].sel(time=TEST)
    bc1, w_bc = bias_corrected_blend(s["hres"], s["pangu"], s["truth"], d, power=1)
    bc2, _    = bias_corrected_blend(s["hres"], s["pangu"], s["truth"], d, power=2)
    s["blend_bc"], s["w_bc"] = bc1, w_bc
    best = min(rmse(s["hres"].sel(time=TEST), tr), rmse(s["pangu"].sel(time=TEST), tr))
    rows.append({
        "lead_day": d,
        "SmartBlend":     rmse(s["blend"].sel(time=TEST), tr),
        "BiasCorr_p1":    rmse(bc1.sel(time=TEST), tr),
        "BiasCorr_p2":    rmse(bc2.sel(time=TEST), tr),
        "best_single":    best,
    })
df_bc = pd.DataFrame(rows)
df_bc["gain_p1_%"] = 100*(1 - df_bc.BiasCorr_p1/df_bc.best_single)
df_bc["gain_p2_%"] = 100*(1 - df_bc.BiasCorr_p2/df_bc.best_single)
print(df_bc.round(3))
df_bc.to_csv(f"{SAVE}/results_biascorrected.csv", index=False)

In [ ]:
d = 5
s = store[d]
tr = s["truth"]

# Shuffle time so any accidental use of "today's truth" would be destroyed.
# Honest method: error should barely change. Leaky method: error would jump.
shuffled = tr.copy()
idx = np.random.permutation(shuffled.sizes["time"])
shuffled_vals = shuffled.isel(time=idx).values
tr_shuf = xr.DataArray(shuffled_vals, coords=tr.coords, dims=tr.dims)

honest, _ = bias_corrected_blend(s["hres"], s["pangu"], tr, d, power=1)
print("honest blend RMSE:", round(rmse(honest.sel(time=TEST), tr.sel(time=TEST)), 3))

# Check the shift is actually in the past
shift = d + 1
print("shift used (days):", shift, "| lead:", d)
print("first non-NaN weight date:", str(s["w_bc"].dropna("time", how="all").time[0].values)[:10])
print("data starts:", str(s["w_bc"].time[0].values)[:10])

In [ ]:
rows = []
for d in LEAD_DAYS:
    s  = store[d]
    tr = s["truth"].sel(time=TEST)
    bc, w = bias_corrected_blend(s["hres"], s["pangu"], s["truth"], d, power=1)
    s["final_blend"], s["final_w"] = bc, w
    rows.append({"lead_day": d,
                 "HRES":  rmse(s["hres"].sel(time=TEST), tr),
                 "Pangu": rmse(s["pangu"].sel(time=TEST), tr),
                 "EqualBlend": rmse(((s["hres"]+s["pangu"])/2).sel(time=TEST), tr),
                 "MausamMix": rmse(bc.sel(time=TEST), tr)})
df_f = pd.DataFrame(rows)
df_f["gain_%"] = 100*(1 - df_f.MausamMix/df_f[["HRES","Pangu"]].min(axis=1))
print(df_f.round(3))
df_f.to_csv(f"{SAVE}/results_final.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5,4.8))
ax.plot(df_f.lead_day, df_f.HRES,       "o-",  color="#1f77b4", label="IFS HRES (physics)")
ax.plot(df_f.lead_day, df_f.Pangu,      "s-",  color="#ff7f0e", label="Pangu-Weather (AI)")
ax.plot(df_f.lead_day, df_f.EqualBlend, "^--", color="#2ca02c", label="Equal-weight blend")
ax.plot(df_f.lead_day, df_f.MausamMix,  "D-",  color="crimson", lw=2.8, label="MausamMix (ours)")
for _, r in df_f.iterrows():
    ax.annotate(f"−{r['gain_%']:.0f}%", (r.lead_day, r.MausamMix),
                textcoords="offset points", xytext=(0,-16), ha="center",
                color="crimson", fontsize=9, fontweight="bold")
ax.set_xlabel("Lead time (days)"); ax.set_ylabel("RMSE (K)")
ax.set_title("MausamMix cuts temperature forecast error over India by 15–18%\n(unseen test data, 2020–2022)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{SAVE}/chart2_final.png", dpi=200)
plt.show()

In [ ]:
d = 5
w = store[d]["final_w"].sel(time=TEST)
seasons = {"Winter (DJF)": [12,1,2], "Pre-monsoon (MAM)": [3,4,5],
           "Monsoon (JJAS)": [6,7,8,9], "Post-monsoon (ON)": [10,11]}

fig, axes = plt.subplots(1, 4, figsize=(16,4.2), constrained_layout=True)
for ax, (name, months) in zip(axes, seasons.items()):
    ws = w.sel(time=w.time.dt.month.isin(months)).mean("time")
    im = ws.plot(ax=ax, x="longitude", y="latitude", vmin=0.2, vmax=0.8,
                 cmap="coolwarm_r", add_colorbar=False)
    ax.set_title(name, fontsize=11); ax.set_xlabel(""); ax.set_ylabel("")
fig.colorbar(im, ax=axes, shrink=0.85,
             label="← more weight on Pangu (AI)   |   more weight on HRES (physics) →")
fig.suptitle("Learned model weights, Day 5 forecasts over India (2020–2022)", fontsize=13)
plt.savefig(f"{SAVE}/chart3_weightmap.png", dpi=200, bbox_inches="tight")
plt.show()

print("Mean weight on HRES by season:")
for name, months in seasons.items():
    print(f"  {name}: {float(w.sel(time=w.time.dt.month.isin(months)).mean()):.3f}")

In [ ]:
w5 = store[5]["final_w"].sel(time=TEST)
mon = w5.sel(time=w5.time.dt.month.isin([6,7,8,9])).mean("time")
win = w5.sel(time=w5.time.dt.month.isin([12,1,2])).mean("time")
diff = mon - win

fig, ax = plt.subplots(figsize=(5.5,5))
im = diff.plot(ax=ax, x="longitude", y="latitude", cmap="PuOr", vmin=-0.2, vmax=0.2,
               cbar_kwargs={"label":"Monsoon minus Winter weight on HRES"})
ax.set_title("Seasonal shift in model trust (Day 5)")
plt.tight_layout(); plt.savefig(f"{SAVE}/chart4_seasonal_shift.png", dpi=200)
plt.show()
print("max shift:", float(abs(diff).max()).__round__(3))

In [ ]:
import xarray as xr, numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/sih081'

out = xr.open_dataset(f"{SAVE}/india_t2m.nc")
LEAD_DAYS = [1, 3, 5, 7, 10]
TEST = slice("2020-01-01", "2022-12-20")

store = {}
for d in LEAD_DAYS:
    store[d] = {}
    store[d]["hres"] = out[f"hres_d{d}"]
    store[d]["pangu"] = out[f"pangu_d{d}"]
    store[d]["truth"] = out[f"truth_d{d}"]

def rmse(f, t):
    w = np.cos(np.deg2rad(f.latitude))
    return float(np.sqrt(((f - t) ** 2).weighted(w).mean()))

def past(x, shift, window=30):
    return x.shift(time=shift).rolling(time=window, min_periods=10).mean()

def bias_corrected_blend(f1, f2, truth, lead_days):
    shift = lead_days + 1
    c1 = f1 - past(f1 - truth, shift)
    c2 = f2 - past(f2 - truth, shift)
    m1 = past((c1 - truth) ** 2, shift)
    m2 = past((c2 - truth) ** 2, shift)
    i1 = 1 / (m1 + 1e-6)
    i2 = 1 / (m2 + 1e-6)
    w1 = i1 / (i1 + i2)
    return w1 * c1 + (1 - w1) * c2, w1

rows = []
for d in LEAD_DAYS:
    s = store[d]
    tr = s["truth"].sel(time=TEST)
    bc, w = bias_corrected_blend(s["hres"], s["pangu"], s["truth"], d)
    s["final_blend"] = bc
    s["final_w"] = w
    eq = (s["hres"] + s["pangu"]) / 2
    r = {}
    r["lead_day"] = d
    r["HRES"] = rmse(s["hres"].sel(time=TEST), tr)
    r["Pangu"] = rmse(s["pangu"].sel(time=TEST), tr)
    r["EqualBlend"] = rmse(eq.sel(time=TEST), tr)
    r["MausamMix"] = rmse(bc.sel(time=TEST), tr)
    rows.append(r)

df_f = pd.DataFrame(rows)
best = df_f[["HRES", "Pangu"]].min(axis=1)
df_f["gain_%"] = 100 * (1 - df_f["MausamMix"] / best)
print(df_f.round(3))

if os.path.exists(f"{SAVE}/results_fair.csv"):
    df_fair = pd.read_csv(f"{SAVE}/results_fair.csv")
    print(df_fair.round(3))
else:
    print("results_fair.csv not found - will type those numbers into slide 4 manually")

In [ ]:
import matplotlib as mpl
from google.colab import files
mpl.rcParams.update({"font.size": 12, "legend.fontsize": 11})
x = df_f["lead_day"]

# FIG 1
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(x, df_f["HRES"], "o-", lw=2, label="IFS HRES (physics)")
ax.plot(x, df_f["Pangu"], "s-", lw=2, label="Pangu-Weather (AI)")
ax.set(xlabel="Lead time (days)", ylabel="RMSE (K)")
ax.set_title("No single model is best: skill gap varies with lead time\n2 m temperature over India, 2020-2022")
ax.legend(); ax.grid(alpha=0.3)
fig.savefig(f"{SAVE}/fig1_model_comparison.png", dpi=300, bbox_inches="tight")

# FIG 2
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, df_f["HRES"], "o-", lw=2, label="IFS HRES (physics)")
ax.plot(x, df_f["Pangu"], "s-", lw=2, label="Pangu-Weather (AI)")
ax.plot(x, df_f["EqualBlend"], "^--", lw=1.8, label="Equal-weight blend")
ax.plot(x, df_f["MausamMix"], "D-", color="crimson", lw=3, label="MausamMix (our blend)")
for i in range(len(df_f)):
    g = df_f["gain_%"][i]
    ax.annotate(f"-{g:.0f}%", (x[i], df_f["MausamMix"][i]), xytext=(0, -20), textcoords="offset points", ha="center", color="crimson", fontweight="bold")
ax.set(xlabel="Lead time (days)", ylabel="RMSE (K)", ylim=(0.55, 2.3))
ax.set_title("MausamMix cuts temperature forecast error over India by 15-18%\nUnseen test data, 2020-2022")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
fig.savefig(f"{SAVE}/fig2_headline_result.png", dpi=300, bbox_inches="tight")

# FIG 3
w5 = store[5]["final_w"].sel(time=TEST)
names = ["Winter (DJF)", "Pre-monsoon (MAM)", "Monsoon (JJAS)", "Post-monsoon (ON)"]
months = [[12, 1, 2], [3, 4, 5], [6, 7, 8, 9], [10, 11]]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, n, m in zip(axes, names, months):
    ws = w5.sel(time=w5.time.dt.month.isin(m)).mean("time")
    im = ws.plot(ax=ax, x="longitude", y="latitude", vmin=0.2, vmax=0.8, cmap="coolwarm_r", add_colorbar=False)
    ax.set(title=n, xlabel="Longitude (E)", ylabel="Latitude (N)")
fig.subplots_adjust(right=0.88, top=0.84, wspace=0.28)
fig.colorbar(im, cax=fig.add_axes([0.9, 0.15, 0.015, 0.65]), label="Weight on HRES (blue) vs Pangu (red)")
fig.suptitle("Learned model weights by region and season - Day 5 forecasts over India", fontsize=14)
fig.savefig(f"{SAVE}/fig3_weight_maps.png", dpi=300, bbox_inches="tight")

# FIG 4
mon = w5.sel(time=w5.time.dt.month.isin([6, 7, 8, 9])).mean("time")
win = w5.sel(time=w5.time.dt.month.isin([12, 1, 2])).mean("time")
fig, ax = plt.subplots(figsize=(6, 5.4))
(mon - win).plot(ax=ax, x="longitude", y="latitude", cmap="PuOr", vmin=-0.22, vmax=0.22, cbar_kwargs={"label": "Monsoon minus Winter weight on HRES"})
ax.set(xlabel="Longitude (E)", ylabel="Latitude (N)")
ax.set_title("Model trust shifts by up to 0.22 between seasons\nin the same location (Day 5)")
fig.savefig(f"{SAVE}/fig4_seasonal_shift.png", dpi=300, bbox_inches="tight")

# TABLES + DOWNLOAD
df_f.round(3).to_csv(f"{SAVE}/table1_results_final.csv", index=False)
df_fair.round(3).to_csv(f"{SAVE}/table2_verification_fairness.csv", index=False)
out_files = ["fig1_model_comparison.png", "fig2_headline_result.png", "fig3_weight_maps.png", "fig4_seasonal_shift.png", "table1_results_final.csv", "table2_verification_fairness.csv"]
for f in out_files:
    files.download(f"{SAVE}/{f}")
plt.show()
print("ALL SAVED:", out_files)

In [ ]:
from google.colab import files

# ---- TABLE 1: final results ----
t1 = df_f.copy()
t1.columns = ["Lead (days)", "IFS HRES", "Pangu-Weather", "Equal blend", "MausamMix", "Gain vs best (%)"]
t1 = t1.round(3)
t1["Gain vs best (%)"] = t1["Gain vs best (%)"].round(1)
t1.to_csv(f"{SAVE}/table1_results_final.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
tb = ax.table(cellText=t1.values, colLabels=t1.columns, loc="center", cellLoc="center")
tb.auto_set_font_size(False); tb.set_fontsize(12); tb.scale(1, 1.6)
for j in range(len(t1.columns)):
    tb[0, j].set_facecolor("#0B1F3A"); tb[0, j].get_text().set_color("white"); tb[0, j].get_text().set_fontweight("bold")
for i in range(1, len(t1) + 1):
    tb[i, 4].set_facecolor("#FDE2E4"); tb[i, 5].set_facecolor("#FDE2E4")
ax.set_title("2 m temperature RMSE (K) over India, test years 2020-2022", fontsize=13, pad=10)
fig.savefig(f"{SAVE}/table1_results_final.png", dpi=300, bbox_inches="tight")

# ---- TABLE 2: verification fairness ----
t2 = df_fair.copy()
t2.columns = ["Lead (days)", "HRES vs ERA5", "HRES vs own analysis", "Pangu vs ERA5"]
t2 = t2.round(3)
t2.to_csv(f"{SAVE}/table2_verification_fairness.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 2.6))
ax.axis("off")
tb = ax.table(cellText=t2.values, colLabels=t2.columns, loc="center", cellLoc="center")
tb.auto_set_font_size(False); tb.set_fontsize(12); tb.scale(1, 1.6)
for j in range(len(t2.columns)):
    tb[0, j].set_facecolor("#0B1F3A"); tb[0, j].get_text().set_color("white"); tb[0, j].get_text().set_fontweight("bold")
ax.set_title("Choice of 'truth' changes model ranking (RMSE, K)", fontsize=13, pad=10)
fig.savefig(f"{SAVE}/table2_verification_fairness.png", dpi=300, bbox_inches="tight")

# ---- FIG 1 re-saved with an accurate title ----
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(df_f["lead_day"], df_f["HRES"], "o-", lw=2, label="IFS HRES (physics)")
ax.plot(df_f["lead_day"], df_f["Pangu"], "s-", lw=2, label="Pangu-Weather (AI)")
ax.set(xlabel="Lead time (days)", ylabel="RMSE (K)")
ax.set_title("Physics and AI models differ in skill, and the gap narrows with lead time\n2 m temperature over India, 2020-2022")
ax.legend(); ax.grid(alpha=0.3)
fig.savefig(f"{SAVE}/fig1_model_comparison.png", dpi=300, bbox_inches="tight")

plt.show()
for f in ["table1_results_final.csv", "table1_results_final.png", "table2_verification_fairness.csv", "table2_verification_fairness.png", "fig1_model_comparison.png"]:
    files.download(f"{SAVE}/{f}")
print("done")

In [ ]:
def table_png(df, cols, title, fname, highlight=()):
    d = df.copy()
    d.columns = cols
    d[cols[0]] = d[cols[0]].astype(int)
    for c in cols[1:]:
        d[c] = d[c].map(lambda v: f"{v:.3f}" if "%" not in c else f"{v:.1f}")
    fig, ax = plt.subplots(figsize=(1.9 * len(cols), 2.8))
    ax.axis("off")
    tb = ax.table(cellText=d.values, colLabels=cols, loc="center", cellLoc="center")
    tb.auto_set_font_size(False)
    tb.set_fontsize(11)
    tb.scale(1, 1.7)
    for j in range(len(cols)):
        tb[0, j].set_facecolor("#0B1F3A")
        tb[0, j].get_text().set_color("white")
        tb[0, j].get_text().set_fontweight("bold")
    for j in highlight:
        for i in range(1, len(d) + 1):
            tb[i, j].set_facecolor("#FDE2E4")
    ax.set_title(title, fontsize=13, pad=12)
    fig.savefig(f"{SAVE}/{fname}", dpi=300, bbox_inches="tight")
    plt.show()

table_png(df_f, ["Lead\n(days)", "IFS HRES", "Pangu", "Equal\nblend", "MausamMix", "Gain vs\nbest (%)"],
          "2 m temperature RMSE (K) over India, test years 2020-2022", "table1_results_final.png", highlight=(4, 5))

table_png(df_fair, ["Lead\n(days)", "HRES vs\nERA5", "HRES vs own\nanalysis", "Pangu vs\nERA5"],
          "Choice of verification truth changes model ranking (RMSE, K)", "table2_verification_fairness.png")

from google.colab import files
files.download(f"{SAVE}/table1_results_final.png")
files.download(f"{SAVE}/table2_verification_fairness.png")

In [ ]:
d = 5
s = store[d]
tr = s["truth"].sel(time=TEST)
h = s["hres"].sel(time=TEST)
p = s["pangu"].sel(time=TEST)
b = s["final_blend"].sel(time=TEST)

w = np.cos(np.deg2rad(tr.latitude))
def daily_rmse(f):
    return np.sqrt(((f - tr) ** 2).weighted(w).mean(["latitude", "longitude"]))

r_h = daily_rmse(h)
r_p = daily_rmse(p)
r_b = daily_rmse(b)
best = np.minimum(r_h, r_p)
gain = 100 * (1 - r_b / best)

monsoon = gain.time.dt.month.isin([6, 7, 8, 9])
g_m = gain.where(monsoon, drop=True)
target = float(g_m.median())
pick = g_m.time[int(abs(g_m - target).argmin())].values
init_str = str(pick)[:10]
valid_str = str(pick + np.timedelta64(d, "D"))[:10]
print("Median-gain monsoon day:", init_str, "| gain:", round(float(gain.sel(time=pick)), 1), "%")

panels = [
    ("IFS HRES (physics)", h.sel(time=pick), float(r_h.sel(time=pick))),
    ("Pangu-Weather (AI)", p.sel(time=pick), float(r_p.sel(time=pick))),
    ("MausamMix (our blend)", b.sel(time=pick), float(r_b.sel(time=pick))),
    ("Actual (ERA5)", tr.sel(time=pick), None),
]

vmin = float(tr.sel(time=pick).min()) - 273.15
vmax = float(tr.sel(time=pick).max()) - 273.15

fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, (name, da, err) in zip(axes, panels):
    im = (da - 273.15).plot(ax=ax, x="longitude", y="latitude", cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False)
    title = name if err is None else f"{name}\nerror: {err:.2f} K"
    ax.set_title(title, fontsize=12, fontweight="bold" if "MausamMix" in name else "normal")
    ax.set_xlabel("Longitude (E)")
    ax.set_ylabel("Latitude (N)")
fig.subplots_adjust(right=0.9, top=0.8, wspace=0.28)
fig.colorbar(im, cax=fig.add_axes([0.915, 0.15, 0.012, 0.6]), label="2 m temperature (°C)")
fig.suptitle(f"Prototype output: Day-5 forecast issued {init_str}, valid {valid_str} (typical monsoon day)", fontsize=14)
fig.savefig(f"{SAVE}/fig5_prototype_output.png", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download(f"{SAVE}/fig5_prototype_output.png")